In [ ]:
# CELL 1: Setup
import sys
sys.path.append('..')

import os
import json
import torch
import numpy as np
import random
import matplotlib.pyplot as plt

from configs.config import Config
from data.splits import get_datasets
from models.bu_net import BUNet
from models.prototypical_segmentation import PrototypicalSegmentation
from training.prototypical_trainer import PrototypicalTrainer

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

Config.create_dirs()
print(f"✓ Device: {Config.DEVICE}")

ModuleNotFoundError: No module named 'torch'

In [ ]:
# CELL 2: Data Loading
train_dataset, val_dataset, test_dataset = get_datasets(Config)

print(f"✓ Train samples: {len(train_dataset)}")
print(f"✓ Val samples:   {len(val_dataset)}")
print(f"✓ Test samples:  {len(test_dataset)}")

In [ ]:
# CELL 3: Create Model + Load Pretrained Encoder
model = PrototypicalSegmentation(
    encoder_name=Config.ENCODER_NAME,
    in_channels=Config.IN_CHANNELS,
    num_classes=Config.NUM_CLASSES,
).to(Config.DEVICE)

# Load pretrained baseline weights (encoder + decoder)
checkpoint = torch.load(
    os.path.join(Config.CHECKPOINT_DIR, 'best_model.pth'),
    weights_only=False,
    map_location=Config.DEVICE,
)
model.unet.load_state_dict(checkpoint['model_state_dict'], strict=False)

print(f"✓ Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("✓ Loaded pretrained weights from baseline")

In [ ]:
# CELL 4: Sanity Check
trainer = PrototypicalTrainer(
    model=model,
    config=Config,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
)

# Test one episode
episode = trainer.sample_episode(train_dataset, k_shot=5, n_query=10)
print(f"✓ Support: {episode['support_images'].shape}")
print(f"✓ Query:   {episode['query_images'].shape}")

# Test prototype extraction
support_imgs = episode['support_images'].to(Config.DEVICE)
support_masks = episode['support_masks'].to(Config.DEVICE)
prototypes = model.compute_prototypes(support_imgs, support_masks)
print(f"✓ Extracted {len(prototypes)} prototypes, shape: {prototypes[0].shape}")

In [ ]:
# CELL 5: Train Prototypical Networks
print("=" * 60)
print("STARTING PROTOTYPICAL NETWORKS TRAINING")
print("=" * 60)

history = trainer.train(
    num_episodes=1000,
    k_shot=5,
    n_query=Config.N_QUERY,
)

# Training Curve
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(history['train_loss'], label='Train Loss')
ax.plot(history['val_loss'], label='Val Loss')
ax.set_xlabel('Episode (×100)')
ax.set_ylabel('Dice Loss')
ax.set_title('Prototypical Networks Training')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(Config.RESULTS_DIR, 'prototypical_training_curve.png'), dpi=150)
plt.show()

In [ ]:
# CELL 6: Evaluate at Different k Values
k_shot_results = trainer.evaluate_k_shot(
    k_values=Config.K_SHOT_VALUES,
    num_episodes=50,
)

# Print summary
print(f"\n{'='*50}")
print("PROTOTYPICAL NETWORKS K-SHOT RESULTS")
print(f"{'='*50}")
for k, result in k_shot_results.items():
    print(f"  k={k:>2}: Dice = {result['mean']:.4f} ± {result['std']:.4f}")
print(f"{'='*50}")

# Save
results_path = os.path.join(Config.RESULTS_DIR, 'prototypical_kshot_results.json')
with open(results_path, 'w') as f:
    json.dump({str(k): {'mean': float(v['mean']), 'std': float(v['std'])}
               for k, v in k_shot_results.items()}, f, indent=4)
print(f"✓ Saved to: {results_path}")

In [ ]:
# CELL 7: Learning Curve
k_values = Config.K_SHOT_VALUES
means = [k_shot_results[k]['mean'] for k in k_values]
stds = [k_shot_results[k]['std'] for k in k_values]

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(k_values, means, yerr=stds, marker='o', capsize=5,
            linewidth=2, markersize=8, color='#4CAF50')
ax.set_xlabel('k (number of support examples)', fontsize=12)
ax.set_ylabel('Mean Tumor Dice Score', fontsize=12)
ax.set_title('Prototypical Networks: Dice vs. k', fontsize=14)
ax.set_xticks(k_values)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

for k, dice in zip(k_values, means):
    ax.text(k, dice + 0.02, f'{dice:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(Config.RESULTS_DIR, 'prototypical_learning_curve.png'), dpi=150)
plt.show()

In [ ]:
# Cell 8: Evaluate at different k values
print("Evaluating few-shot performance...")

k_shot_results = trainer.evaluate_k_shot(
    k_values=[1, 5, 10, 20],
    num_episodes=20  # 20 episodes per k value
)

# Print results
print("\nFew-Shot Performance:")
print("="*40)
for k, result in k_shot_results.items():
    print(f"k={k}: DICE = {result['mean']:.4f} ± {result['std']:.4f}")

# Save results
import json
with open(os.path.join(Config.RESULTS_DIR, 'prototypical_results.json'), 'w') as f:
    json.dump({k: {'mean': float(v['mean']), 'std': float(v['std'])} 
               for k, v in k_shot_results.items()}, f, indent=2)

In [ ]:
# Cell 9: Visualize learning curve
import matplotlib.pyplot as plt

k_values = [1, 5, 10, 20]
mean_dice = [k_shot_results[k]['mean'] for k in k_values]
std_dice = [k_shot_results[k]['std'] for k in k_values]

plt.figure(figsize=(10, 6))
plt.errorbar(k_values, mean_dice, yerr=std_dice, marker='o', capsize=5, 
             linewidth=2, markersize=8)
plt.xlabel('Number of Support Examples (k)', fontsize=12)
plt.ylabel('DICE Score', fontsize=12)
plt.title('Few-Shot Learning Curve: Prototypical Networks', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(k_values)
plt.ylim([0, 1])

# Add values on plot
for k, dice in zip(k_values, mean_dice):
    plt.text(k, dice + 0.02, f'{dice:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(Config.RESULTS_DIR, 'prototypical_learning_curve.png'), dpi=150)
plt.show()

print("✓ Learning curve saved!")

In [ ]:
# Cell 10: Compare with baseline fine-tuning
def baseline_finetune_k_shot(base_model, train_dataset, test_loader, k=5):
    """Baseline: fine-tune on k examples"""
    import copy
    
    model = copy.deepcopy(base_model)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
    loss_fn = smp.losses.DiceLoss(mode='multiclass')
    
    # Sample k examples
    indices = random.sample(range(len(train_dataset)), k)
    
    # Fine-tune
    model.train()
    for epoch in range(20):
        for idx in indices:
            sample = train_dataset[idx]
            img = sample['image'].unsqueeze(0).to(Config.DEVICE)
            mask = sample['mask'].unsqueeze(0).to(Config.DEVICE)
            
            optimizer.zero_grad()
            seg_out, _ = model(img)
            loss = loss_fn(seg_out, mask)
            loss.backward()
            optimizer.step()
    
    # Evaluate (simplified)
    model.eval()
    val_episode = trainer.sample_episode(val_dataset, k_shot=0, n_query=20)
    query_imgs = val_episode['query_images'].to(Config.DEVICE)
    query_masks = val_episode['query_masks'].to(Config.DEVICE)
    
    with torch.no_grad():
        pred, _ = model(query_imgs)
        dice_loss = loss_fn(pred, query_masks)
        dice = 1 - dice_loss.item()
    
    return dice

# Load your baseline model
from models.bu_net import BUNetMultiTask
baseline_model = BUNetMultiTask(
    encoder_name='resnet34',
    in_channels=2,
    num_classes=4,
    num_tumor_types=5
).to(Config.DEVICE)

checkpoint = torch.load('./checkpoints/best_model.pth')
baseline_model.load_state_dict(checkpoint['model_state_dict'])

# Compare baseline vs prototypical
print("\nComparing Prototypical vs Baseline Fine-tuning:")
print("="*50)

baseline_results = {}
for k in [1, 5, 10]:
    dice = baseline_finetune_k_shot(baseline_model, train_dataset, None, k=k)
    baseline_results[k] = dice
    print(f"Baseline k={k}: DICE = {dice:.4f}")
    print(f"Prototypical k={k}: DICE = {k_shot_results[k]['mean']:.4f}")
    improvement = k_shot_results[k]['mean'] - dice
    print(f"  → Improvement: {improvement:+.4f}\n")